# White-box attack evaluation

This notebook evaluates clean and adversarial accuracy for FGSM/PGD attacks.

In [ ]:
# Imports
from py_compile import main

import argparse
import csv
import gc
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms.functional as F

import utils

In [ ]:
# Configuration
# Update BASE_DIR if the notebook is not in the same folder as the data/checkpoints.
BASE_DIR = os.getcwd()

ATTACK = "FGSM"  # Choose: "FGSM" or "PGD"

# Choose which checkpoint to load.
USE_LATENT_ADV = False
USE_INPUT_LATENT_ADV = False
USE_INPUT_ADV = False

batch_size_test = 64
num_workers = 0
num_classes = 43
eps_list_255 = [2, 4, 8, 16]
eps_list = [e / 255.0 for e in eps_list_255]
num_steps = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load testing data
print("Reading testing data")

test_path = os.path.join(BASE_DIR, "GTSRB", "Test", "Final_Test", "Images")
testImages, testLabels = utils.readTrafficSigns_test(test_path)

test_dataset = utils.GTSRBDataset(testImages, testLabels, target_size=224)
print(f"Total testing samples: {len(test_dataset)}")

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size_test,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
)

In [ ]:
# Build model
model = utils.ResNet18_L3(num_classes=num_classes)

if USE_LATENT_ADV:
    checkpoint_name = "resnet18_gtsrb_latent_adv.pth"
elif USE_INPUT_LATENT_ADV:
    checkpoint_name = "resnet18_gtsrb_input_latent_adv.pth"
elif USE_INPUT_ADV:
    checkpoint_name = "resnet18_gtsrb_input_adv.pth"
else:
    checkpoint_name = "resnet18_gtsrb_clean_20.pth"

checkpoint_path = os.path.join(BASE_DIR, checkpoint_name)
print(f"Loading checkpoint: {checkpoint_path}")

state = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state)

model = model.to(device)
model.eval()

criterion = nn.CrossEntropyLoss()

In [ ]:
# Clean accuracy
clean_accuracy, predictions_clean = utils.evaluate_clean_accuracy(
    model,
    test_loader,
    device,
)

print(f"Latent adversarially trained model: {USE_LATENT_ADV}")
print(f"Input + latent adversarially trained model: {USE_INPUT_LATENT_ADV}")
print(f"Input adversarially trained model: {USE_INPUT_ADV}")
print(f"Attack: {ATTACK}")
print(f"Clean accuracy: {clean_accuracy * 100:.2f}%")

In [ ]:
# White-box adversarial evaluation
results = []

for eps in eps_list:
    epsilon = eps
    alpha = epsilon / 4.0

    print(f"\nGenerating adversarial examples with {ATTACK} attack (epsilon={epsilon:.4f})")

    examples_adv, true_labels = utils.generate_adversarial_examples_batched(
        model=model,
        test_loader=test_loader,
        attack=ATTACK,
        device=device,
        epsilon=epsilon,
        criterion=criterion,
        alpha=alpha,
        num_steps=num_steps,
    )

    adv_dataset = TensorDataset(examples_adv, true_labels)
    adv_loader = DataLoader(
        adv_dataset,
        batch_size=batch_size_test,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False,
    )

    adv_accuracy, predictions_adv, wrong_indices, wrong_true_labels, wrong_pred_labels = (
        utils.evaluate_adversarial_accuracy(
            model,
            adv_loader,
            device,
        )
    )

    print(f"Adversarial accuracy: {adv_accuracy * 100:.2f}%")
    print(f"Number of wrong predictions: {len(wrong_indices)}")

    for i in range(min(10, len(wrong_indices))):
        print(f"Index {wrong_indices[i]}: true={wrong_true_labels[i]}, pred={wrong_pred_labels[i]}")

    asr, successful, initially_correct = utils.compute_attack_success_rate(
        predictions_clean,
        predictions_adv,
        true_labels,
    )

    print(f"Attack Success Rate: {asr * 100:.2f}%")
    print(f"Successful attacks: {successful}")
    print(f"Initially correct samples: {initially_correct}")

    results.append(
        {
            "epsilon": epsilon,
            "epsilon_255": epsilon * 255,
            "attack": ATTACK,
            "clean_accuracy": clean_accuracy,
            "adversarial_accuracy": adv_accuracy,
            "attack_success_rate": asr,
            "successful_attacks": successful,
            "initially_correct_samples": initially_correct,
            "num_wrong_predictions": len(wrong_indices),
        }
    )

    del adv_loader, adv_dataset, examples_adv, true_labels
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results